# Create mosaic map tiles layers & API endpoints for LVISF2 L3 gridded data
### for data at MAAP & any DAAC
from TiTiler for mosaic json of map tiles

In [1]:
pip install cogeo-mosaic

  Using cached cogeo_mosaic-8.2.0-py3-none-any.whl.metadata (19 kB)
  Using cached supermorecado-0.1.2-py3-none-any.whl.metadata (8.2 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
Using cached cogeo_mosaic-8.2.0-py3-none-any.whl (40 kB)
Using cached supermorecado-0.1.2-py3-none-any.whl (14 kB)
Using cached typing_inspection-0.4.1-py3-none-any.whl (14 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
from cogeo_mosaic.mosaic import MosaicJSON
from cogeo_mosaic.backends.s3 import S3Backend
import httpx
import json

In [3]:
import contextily as ctx

ModuleNotFoundError: No module named 'contextily'

In [4]:
import sys
sys.path.append('/projects/code/icesat2_boreal/lib')

In [5]:
import mosaiclib
from mosaiclib import *
import maplib_folium

### Create a tiles layer from a dataset at ORNL DAAC  

Using tool developed for NASA MAAP (Multi-mission Algorithm & Analysis Platform; www.maap-project.org):  
+ `Federated Collection Discovery` tool to find `short name` of DAAC dataset of interest:  
https://discover.maap-project.org/

In [8]:
import importlib
importlib.reload(maplib_folium)

<module 'maplib_folium' from '/projects/code/icesat2_boreal/lib/maplib_folium.py'>

In [7]:
from cmr import GranuleQuery

def get_s3_links(SHORT_NAME, filter_string=None):
    search = GranuleQuery(mode="https://cmr.earthdata.nasa.gov/search/").short_name(
        SHORT_NAME
    )
    granules = search.get_all()
    
    s3_links = []
    
    for granule in granules:
        s3_link = next(
            filter(lambda d: "download access via S3" in d.get("title"), granule["links"]),
            None,
        )
        assert s3_link
        s3_links.append(s3_link["href"])

    if filter_string is not None:
        s3_links = [f for f in s3_links if filter_string in f]

    return s3_links

In [19]:
%%time
decid_frac_2015_tiles_layer_dict = maplib_folium.make_tiles_layer_dict(
                                            None, 
                                            s3_links_decid_frac, 
                                            "Decid. fraction prediction 2015", 
                                            SHOW_CBAR=True, 
                                            PARAMS_DICT = {"rescale": f"0,100", "bidx":"1", "colormap_name": "cividis"},
    PRINT=True
                                           )
decid_frac_2015_tiles_layer_dict['layer'].tiles



{'rescale': '0,100', 'bidx': '1', 'colormap_name': 'cividis'}




{'rescale': '0,100', 'bidx': '1', 'colormap_name': 'cividis'}


CMAP unaltered: cividis
tiles with CMAP changed: https://titiler-pgstac.maap-project.org/mosaics/3f1702d9-8261-4018-853b-f1cefbae4d80/tiles/{z}/{x}/{y}@1x?rescale=0,100&bidx=1&colormap_name=cividis
Decid. fraction prediction 2015 tiles (cividis): https://titiler-pgstac.maap-project.org/mosaics/3f1702d9-8261-4018-853b-f1cefbae4d80/tiles/{z}/{x}/{y}@1x?rescale=0,100&bidx=1&colormap_name=cividis
CPU times: user 81.2 ms, sys: 1.8 ms, total: 83 ms
Wall time: 1.5 s


'https://titiler-pgstac.maap-project.org/mosaics/3f1702d9-8261-4018-853b-f1cefbae4d80/tiles/{z}/{x}/{y}@1x?rescale=0,100&bidx=1&colormap_name=cividis'

### Here is the top level function to create the API endpoint for an ORNL DAAC dataset

In [8]:
# preview the S3 links
s3_links_LVISF2_RH98 = get_s3_links("ABoVE_LVIS_VegetationStructure_1923", filter_string='RH098_mean_30m.tif' )
s3_links_LVISF2_RH98

['s3://ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_056233_RH098_mean_30m.tif',
 's3://ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_056593_RH098_mean_30m.tif',
 's3://ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_057198_RH098_mean_30m.tif',
 's3://ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_057748_RH098_mean_30m.tif',
 's3://ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_058064_RH098_mean_30m.tif',
 's3://ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_058321_RH098_mean_30m.tif',
 's3://ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_058625_RH098_mean_30m.tif',
 's3://ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructu

In [ ]:
%%time
LVISF2_RH98_tiles_layer_dict = maplib_folium.make_tiles_layer_dict(
                                            None, 
                                            s3_links_LVISF2_RH98, 
                                            "ABoVE LVISF2 L3 gridded RH98 [30m]", 
                                            SHOW_CBAR=True, 
                                            PARAMS_DICT = {"rescale": f"0,20", "bidx":"1", "colormap_name": "inferno"},
    PRINT=True
                                           )
LVISF2_RH98_tiles_layer_dict['layer'].tiles



{'rescale': '0,20', 'bidx': '1', 'colormap_name': 'inferno'}




/opt/conda/envs/pangeo/lib/python3.12/site-packages/rio_tiler/io/rasterio.py:135: NoOverviewWarning: The dataset has no Overviews. rio-tiler performances might be impacted.
  warnings.warn(
